# CRM Access Governance & Customer Data Protection
## Stage 4 — Data Quality Framework

This notebook extends the project from access governance into a formal **Data Quality Framework**.

### Main Goals
1. Define relevant data quality dimensions.
2. Build a reusable Data Quality Rule Catalog.
3. Execute rules against the CRM access dataset.
4. Measure pass/fail rates.
5. Identify records that violate quality expectations.
6. Calculate quality scores by rule and dimension.
7. Distinguish data quality issues from legitimate security anomalies.
8. Prepare a quality-monitoring layer for future Power BI reporting.

> **Important:** not every standard data quality dimension can be fully measured with this dataset. Unsupported dimensions are documented rather than artificially inferred.


In [ ]:
# 1. Libraries and Settings
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_PATH = Path("Permission_Aware_CRM_Governance_Synthetic_50000.csv")
print("Environment ready.")


# 2. Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
df_dq = df.copy()

print(f"Rows: {df_dq.shape[0]:,}")
print(f"Columns: {df_dq.shape[1]}")
display(df_dq.head())


# 3. Data Quality Dimensions

This project uses:
- **Completeness**
- **Validity**
- **Uniqueness**
- **Consistency**
- **Timeliness**
- **Accuracy**

Timeliness is limited because the dataset has no complete timestamp. Accuracy cannot be fully validated because there is no external source of truth.


# 4. Initial Data Profiling

In [ ]:
profile = pd.DataFrame({
    "Data_Type": df_dq.dtypes.astype(str),
    "Non_Null": df_dq.notna().sum(),
    "Nulls": df_dq.isna().sum(),
    "Null_%": (df_dq.isna().mean() * 100).round(2),
    "Unique_Values": df_dq.nunique(),
    "Cardinality_%": (df_dq.nunique() / len(df_dq) * 100).round(2)
})
display(profile)


# 5. Approved Reference Domains

In [ ]:
approved_roles = {"Admin","Manager","Sales Rep","Support","Analyst"}
approved_actions = {"ViewLead","EditLead","CreateOpportunity","ApproveDiscount","ExportCRM","DeleteLead"}
approved_device_types = {"Managed","BYOD"}
print("Reference domains prepared.")


# 6. Data Quality Rule Catalog

In [ ]:
dq_rule_catalog = pd.DataFrame([
    ["DQ-COMP-001","Completeness","User_ID","User_ID must not be null.","Critical","Access events require a traceable user identifier."],
    ["DQ-COMP-002","Completeness","Role","Role must not be null.","Critical","Role is required for access-control evaluation."],
    ["DQ-COMP-003","Completeness","CRM_Action","CRM_Action must not be null.","Critical","Requested action is required for authorization evaluation."],
    ["DQ-COMP-004","Completeness","Permission_Granted","Permission_Granted must not be null.","Critical","Explicit permission is required for governance logic."],
    ["DQ-COMP-005","Completeness","Access_Decision","Access_Decision must not be null.","High","Outcome is required for monitoring."],
    ["DQ-VAL-001","Validity","Role","Role must belong to the approved role domain.","Critical","Unknown roles cannot be evaluated reliably."],
    ["DQ-VAL-002","Validity","CRM_Action","CRM_Action must belong to the approved action domain.","Critical","Unknown actions may bypass controls."],
    ["DQ-VAL-003","Validity","Device_Type","Device_Type must be Managed or BYOD.","High","Device context is used in risk evaluation."],
    ["DQ-VAL-004","Validity","Access_Hour","Access_Hour must be between 0 and 23.","High","Invalid hour values compromise temporal analysis."],
    ["DQ-VAL-005","Validity","Data_Sensitivity","Data_Sensitivity must be between 1 and 5.","Critical","Sensitivity affects governance decisions."],
    ["DQ-VAL-006","Validity","Anomaly_Score","Anomaly_Score must be between 0 and 1.","High","Out-of-range values invalidate risk interpretation."],
    ["DQ-VAL-007","Validity","Policy_Compliance_Score","Policy_Compliance_Score must be between 0 and 100.","High","Score must remain within its defined scale."],
    ["DQ-VAL-008","Validity","Governance_Score","Governance_Score must be between 0 and 100.","High","Score must remain within its defined scale."],
    ["DQ-VAL-009","Validity","Daily_Logins","Daily_Logins must be zero or greater.","Medium","Negative login counts are invalid."],
    ["DQ-VAL-010","Validity","Failed_Logins","Failed_Logins must be zero or greater.","Medium","Negative failed-login counts are invalid."],
    ["DQ-UNI-001","Uniqueness","Full Record","Exact duplicate rows should not exist.","Medium","Duplicate events may distort KPIs."],
    ["DQ-CON-001","Consistency","Role + CRM_Action","Every Role × CRM_Action combination must exist in the governance access matrix.","Critical","Unmapped combinations cannot be evaluated."],
    ["DQ-CON-002","Consistency","Permission_Granted + Access_Decision","Records without permission should not result in a non-blocked synthetic decision.","High","Permission appears to behave as a gate."]
], columns=["Rule_ID","Dimension","Field","Rule_Description","Severity","Rationale"])
display(dq_rule_catalog)


# 7. Governance Access Matrix for Consistency Checks

In [ ]:
roles = ["Admin","Manager","Sales Rep","Support","Analyst"]
actions = ["ViewLead","EditLead","CreateOpportunity","ApproveDiscount","ExportCRM","DeleteLead"]

access_matrix = pd.DataFrame(index=roles, columns=actions)
access_matrix.loc["Admin"] = ["ALLOW"]*6
access_matrix.loc["Manager"] = ["ALLOW","ALLOW","ALLOW","ALLOW","REVIEW","REVIEW"]
access_matrix.loc["Sales Rep"] = ["ALLOW","ALLOW","ALLOW","REVIEW","REVIEW","BLOCK"]
access_matrix.loc["Support"] = ["ALLOW","REVIEW","BLOCK","BLOCK","BLOCK","BLOCK"]
access_matrix.loc["Analyst"] = ["ALLOW","BLOCK","BLOCK","BLOCK","REVIEW","BLOCK"]

access_matrix_long = (
    access_matrix.reset_index()
    .rename(columns={"index":"Role"})
    .melt(id_vars="Role", var_name="CRM_Action", value_name="Baseline_Authorization")
)
display(access_matrix)


# 8. Rule Execution Functions

In [ ]:
def rule_not_null(data, column):
    return data[column].notna()

def rule_in_domain(data, column, allowed_values):
    return data[column].isin(allowed_values)

def rule_between(data, column, lower, upper):
    return data[column].between(lower, upper, inclusive="both")

def rule_non_negative(data, column):
    return data[column].ge(0)

def rule_no_exact_duplicate(data):
    return ~data.duplicated(keep=False)


# 9. Execute Data Quality Rules

In [ ]:
rule_results = {}

rule_results["DQ-COMP-001"] = rule_not_null(df_dq, "User_ID")
rule_results["DQ-COMP-002"] = rule_not_null(df_dq, "Role")
rule_results["DQ-COMP-003"] = rule_not_null(df_dq, "CRM_Action")
rule_results["DQ-COMP-004"] = rule_not_null(df_dq, "Permission_Granted")
rule_results["DQ-COMP-005"] = rule_not_null(df_dq, "Access_Decision")

rule_results["DQ-VAL-001"] = rule_in_domain(df_dq, "Role", approved_roles)
rule_results["DQ-VAL-002"] = rule_in_domain(df_dq, "CRM_Action", approved_actions)
rule_results["DQ-VAL-003"] = rule_in_domain(df_dq, "Device_Type", approved_device_types)
rule_results["DQ-VAL-004"] = rule_between(df_dq, "Access_Hour", 0, 23)
rule_results["DQ-VAL-005"] = rule_between(df_dq, "Data_Sensitivity", 1, 5)
rule_results["DQ-VAL-006"] = rule_between(df_dq, "Anomaly_Score", 0, 1)
rule_results["DQ-VAL-007"] = rule_between(df_dq, "Policy_Compliance_Score", 0, 100)
rule_results["DQ-VAL-008"] = rule_between(df_dq, "Governance_Score", 0, 100)
rule_results["DQ-VAL-009"] = rule_non_negative(df_dq, "Daily_Logins")
rule_results["DQ-VAL-010"] = rule_non_negative(df_dq, "Failed_Logins")

rule_results["DQ-UNI-001"] = rule_no_exact_duplicate(df_dq)

mapped_pairs = set(zip(access_matrix_long["Role"], access_matrix_long["CRM_Action"]))
rule_results["DQ-CON-001"] = pd.Series(
    [(r,a) in mapped_pairs for r,a in zip(df_dq["Role"], df_dq["CRM_Action"])],
    index=df_dq.index
)
rule_results["DQ-CON-002"] = ~(
    (df_dq["Permission_Granted"] == False) &
    (df_dq["Access_Decision"] != "Block")
)

print("All data quality rules executed.")


# 10. Rule-Level Results

In [ ]:
rows = []
for rule_id, passed_mask in rule_results.items():
    checked = len(passed_mask)
    passed = int(passed_mask.sum())
    failed = checked - passed
    rows.append({
        "Rule_ID": rule_id,
        "Records_Checked": checked,
        "Passed_Records": passed,
        "Failed_Records": failed,
        "Pass_Rate": passed / checked * 100,
        "Fail_Rate": failed / checked * 100
    })

dq_rule_results = pd.DataFrame(rows).merge(dq_rule_catalog, on="Rule_ID", how="left")

display(
    dq_rule_results[
        ["Rule_ID","Dimension","Field","Severity","Records_Checked",
         "Passed_Records","Failed_Records","Pass_Rate","Fail_Rate",
         "Rule_Description","Rationale"]
    ].sort_values(["Dimension","Fail_Rate"], ascending=[True,False]).round(3)
)


# 11. Overall Data Quality Score

In [ ]:
severity_weights = {"Critical":4,"High":3,"Medium":2,"Low":1}
dq_rule_results["Severity_Weight"] = dq_rule_results["Severity"].map(severity_weights)

overall_dq_score = np.average(
    dq_rule_results["Pass_Rate"],
    weights=dq_rule_results["Severity_Weight"]
)

print(f"Overall Data Quality Score: {overall_dq_score:.2f}%")


# 12. Data Quality Score by Dimension

In [ ]:
dimension_rows = []
for dimension, group in dq_rule_results.groupby("Dimension"):
    dimension_rows.append({
        "Dimension": dimension,
        "Rules": len(group),
        "Failed_Records": int(group["Failed_Records"].sum()),
        "Weighted_Pass_Rate": np.average(
            group["Pass_Rate"],
            weights=group["Severity_Weight"]
        )
    })

dimension_score = pd.DataFrame(dimension_rows)
display(dimension_score.sort_values("Weighted_Pass_Rate").round(3))


# 13. Record-Level Quality Flags

In [ ]:
dq_record_flags = pd.DataFrame(
    {rule_id: mask.astype(int) for rule_id, mask in rule_results.items()},
    index=df_dq.index
)

rule_columns = list(rule_results.keys())
dq_record_flags["Failed_Rule_Count"] = len(rule_columns) - dq_record_flags[rule_columns].sum(axis=1)

def quality_status(failed_count):
    if failed_count == 0:
        return "PASS"
    elif failed_count <= 2:
        return "WARNING"
    return "FAIL"

dq_record_flags["DQ_Status"] = dq_record_flags["Failed_Rule_Count"].apply(quality_status)

display(dq_record_flags.head())
display(dq_record_flags["DQ_Status"].value_counts().to_frame("Records"))


# 14. Governed Monitoring Dataset

In [ ]:
df_dq_monitoring = df_dq.copy()
df_dq_monitoring = df_dq_monitoring.join(
    dq_record_flags[["Failed_Rule_Count","DQ_Status"]]
)

display(
    df_dq_monitoring[
        ["User_ID","Role","CRM_Action","Permission_Granted",
         "Access_Decision","Failed_Rule_Count","DQ_Status"]
    ].head(20)
)


# 15. Failed Rule Detail Table

In [ ]:
failed_rule_rows = []
for rule_id, passed_mask in rule_results.items():
    for idx in passed_mask.index[~passed_mask]:
        failed_rule_rows.append({
            "Record_Index": idx,
            "User_ID": df_dq.loc[idx, "User_ID"],
            "Rule_ID": rule_id
        })

dq_failures = pd.DataFrame(
    failed_rule_rows,
    columns=["Record_Index","User_ID","Rule_ID"]
)

if not dq_failures.empty:
    dq_failures = dq_failures.merge(
        dq_rule_catalog[["Rule_ID","Dimension","Field","Severity","Rule_Description"]],
        on="Rule_ID",
        how="left"
    )

display(dq_failures.head(30))


# 16. Rule Prioritization

Prototype remediation priority:

`Severity Weight × Fail Rate`


In [ ]:
dq_rule_results["Remediation_Priority_Score"] = (
    dq_rule_results["Severity_Weight"] * dq_rule_results["Fail_Rate"]
)

priority_table = dq_rule_results[
    ["Rule_ID","Dimension","Field","Severity","Fail_Rate",
     "Remediation_Priority_Score","Rule_Description"]
].sort_values("Remediation_Priority_Score", ascending=False)

display(priority_table.round(3))


# 17. Supported vs. Unsupported Dimensions

### Supported
- Completeness
- Validity
- Uniqueness
- Consistency

### Limited
- Timeliness

### Not Reliably Supported
- Accuracy

A mature framework should explicitly document dimensions that cannot be measured rather than fabricate evidence.


# 18. Proposed Data Quality Operating Model

| Responsibility | Example Role |
|---|---|
| Define business quality expectations | Data Owner |
| Maintain rules and definitions | Data Steward |
| Implement technical checks | Data Engineering |
| Monitor security-related failures | Information Security |
| Review privacy-related controls | Privacy / DPO |
| Consume quality dashboards | Business & Analytics Teams |


# 19. Data Quality Monitoring KPIs

Future Power BI metrics:
- Overall Data Quality Score
- Pass Rate by Dimension
- Failed Records
- Critical Rule Failures
- High-Severity Rule Failures
- Failed Rules by Field
- Failed Rules by Role
- Failed Rules by CRM Action
- Records with Multiple Failures
- Data Quality Trend Over Time


# 20. Exportable Governance Artifacts

Reusable DataFrames:
- `dq_rule_catalog`
- `dq_rule_results`
- `dq_record_flags`
- `dq_failures`
- `df_dq_monitoring`
- `priority_table`


# 21. Findings to Document

## 21.1 Completeness
- Missing required fields:
- Critical identifier issues:

## 21.2 Validity
- Invalid fields:
- Domain violations:

## 21.3 Uniqueness
- Exact duplicate events:
- Need for a business event key:

## 21.4 Consistency
- Unmapped role-action combinations:
- Permission/decision inconsistencies:

## 21.5 Overall Quality
- Overall DQ Score:
- Lowest-scoring dimension:
- Highest-priority rule:
- Records requiring remediation:

## 21.6 Governance Conclusions
1.
2.
3.


# 22. Limitations

1. The dataset is synthetic.
2. No complete event timestamp exists.
3. No authoritative external source exists for accuracy validation.
4. The dataset has no explicit business event key.
5. Rule severity levels are proposed for this project.
6. Quality thresholds should be validated with business owners in a real environment.
7. A 100% pass rate does not necessarily imply that the data is accurate or fit for every use case.


# 23. Next Step — Metadata & Data Catalog

## Stage 5 — Metadata & Data Catalog

Planned outputs:
- business glossary;
- technical metadata inventory;
- data domain classification;
- owner and steward fields;
- sensitivity classification;
- field-level governance metadata;
- catalog-ready data dictionary;
- relationship between data fields and governance rules.
